In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
import pandas as pd
from statsbombpy import sb
from mplsoccer import Pitch
import numpy as np

# ── Color system ─────────────────────────────────────────────────────────────
C_BG     = '#0d1117'
C_CARD   = '#161b22'
C_BORDER = '#30363d'
C_ARG    = '#75AADB'
C_FRA    = '#EF3340'
C_TEXT   = '#e6edf3'
C_MUTED  = '#8b949e'
C_GOLD   = '#f0c040'
C_SILVER = '#8b949e'
C_BRONZE = '#cd7f32'

# ── Load match data (2022 WC Final: Argentina 3-3 France) ────────────────────
events = sb.events(match_id=3869685)

# ── Top-level aggregations ────────────────────────────────────────────────────
shots = events[events['type'] == 'Shot']
shots = shots[shots['period'] != 5]
xg    = shots.groupby('team')['shot_statsbomb_xg'].sum().round(2)

possession     = events[events['period'] != 5].groupby('possession_team')['id'].count()
possession_pct = (possession / possession.sum() * 100).round(1)

shots_on_target = (shots[shots['shot_outcome'].isin(['Goal', 'Saved'])]
                   .groupby('team')['id'].count())


# ── Shared panel setup ────────────────────────────────────────────────────────
def _panel(ax, title=None, accent=None):
    ax.set_facecolor(C_CARD)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    rect = mpatches.FancyBboxPatch(
        (0.01, 0.01), 0.98, 0.98, boxstyle='round,pad=0.01',
        linewidth=1, edgecolor=C_BORDER, facecolor='none',
        transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    if title:
        ax.set_title(title, color=C_MUTED, fontsize=9, pad=8,
                     fontweight='bold', loc='left', x=0.04)
    if accent:
        ax.axhline(y=0.965, xmin=0.04, xmax=0.18,
                   color=accent, linewidth=2.5, transform=ax.transAxes)


# ── Match stats with split comparison bars ────────────────────────────────────
def plot_match_stats(events, team1, team2, ax):
    passes     = events[events['type'] == 'Pass'].groupby('team')['id'].count()
    fouls      = events[events['type'] == 'Foul Committed'].groupby('team')['id'].count()
    corners    = events[(events['type'] == 'Pass') &
                        (events['pass_type'] == 'Corner')].groupby('team')['id'].count()
    recoveries = events[events['type'] == 'Ball Recovery'].groupby('team')['id'].count()
    dribbles   = events[(events['type'] == 'Dribble') &
                        (events['dribble_outcome'] == 'Complete')].groupby('team')['id'].count()

    _panel(ax, 'MATCH STATS', accent=C_MUTED)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.text(0.08, 0.90, team1, color=C_ARG, fontsize=9,
            fontweight='bold', transform=ax.transAxes)
    ax.text(0.92, 0.90, team2, color=C_FRA, fontsize=9,
            fontweight='bold', ha='right', transform=ax.transAxes)
    ax.axhline(y=0.86, xmin=0.04, xmax=0.96,
               color=C_BORDER, linewidth=0.8, transform=ax.transAxes)

    stats = [
        ('Passes',     passes.get(team1, 0),     passes.get(team2, 0)),
        ('Fouls',      fouls.get(team1, 0),      fouls.get(team2, 0)),
        ('Corners',    corners.get(team1, 0),    corners.get(team2, 0)),
        ('Recoveries', recoveries.get(team1, 0), recoveries.get(team2, 0)),
        ('Dribbles',   dribbles.get(team1, 0),   dribbles.get(team2, 0)),
    ]
    bar_ys = [0.75, 0.62, 0.49, 0.36, 0.23]
    L, R   = 0.05, 0.95

    for (label, v1, v2), y in zip(stats, bar_ys):
        total = (v1 + v2) or 1
        split = L + (R - L) * (v1 / total)

        ax.axhline(y=y, xmin=L, xmax=R,
                   color=C_BORDER, linewidth=7, solid_capstyle='butt',
                   transform=ax.transAxes)
        ax.axhline(y=y, xmin=L, xmax=split,
                   color=C_ARG, linewidth=7, solid_capstyle='butt',
                   transform=ax.transAxes)
        ax.axhline(y=y, xmin=split, xmax=R,
                   color=C_FRA, linewidth=7, solid_capstyle='butt',
                   transform=ax.transAxes)

        ax.text(0.08, y + 0.055, str(int(v1)),
                color=C_ARG, fontsize=9, fontweight='bold',
                ha='center', transform=ax.transAxes)
        ax.text(0.5,  y + 0.055, label,
                color=C_MUTED, fontsize=7.5, ha='center', transform=ax.transAxes)
        ax.text(0.92, y + 0.055, str(int(v2)),
                color=C_FRA, fontsize=9, fontweight='bold',
                ha='center', transform=ax.transAxes)


# ── xG flow with filled areas and goal markers ────────────────────────────────
def plot_xgflow(events, team1, team2, ax):
    shots = events[events['type'] == 'Shot'].copy()
    shots = shots[shots['period'] != 5]

    t1 = shots[shots['team'] == team1].sort_values('minute').copy()
    t2 = shots[shots['team'] == team2].sort_values('minute').copy()
    t1['cumxg'] = t1['shot_statsbomb_xg'].cumsum()
    t2['cumxg'] = t2['shot_statsbomb_xg'].cumsum()

    t1_goals = t1[t1['shot_outcome'] == 'Goal']
    t2_goals = t2[t2['shot_outcome'] == 'Goal']

    ymax = max(t1['cumxg'].max(), t2['cumxg'].max()) * 1.18

    ax.set_facecolor(C_BG)
    ax.set_ylim(0, ymax)

    ax.fill_between(t1['minute'], t1['cumxg'], step='post', color=C_ARG, alpha=0.10)
    ax.fill_between(t2['minute'], t2['cumxg'], step='post', color=C_FRA, alpha=0.10)

    ax.step(t1['minute'], t1['cumxg'], where='post',
            color=C_ARG, linewidth=2.5, label=team1, zorder=3)
    ax.step(t2['minute'], t2['cumxg'], where='post',
            color=C_FRA, linewidth=2.5, label=team2, zorder=3)

    for _, g in t1_goals.iterrows():
        cum = t1.loc[t1['minute'] <= g['minute'], 'cumxg'].max()
        ax.scatter(g['minute'], cum, color=C_ARG, s=90, zorder=5, marker='*')
    for _, g in t2_goals.iterrows():
        cum = t2.loc[t2['minute'] <= g['minute'], 'cumxg'].max()
        ax.scatter(g['minute'], cum, color=C_FRA, s=90, zorder=5, marker='*')

    ax.axvline(x=45, color=C_BORDER, linestyle='--', linewidth=1, alpha=0.6)
    ax.text(45.5, ymax * 0.92, 'HT', color=C_MUTED, fontsize=7)
    ax.axvline(x=90, color=C_BORDER, linestyle='--', linewidth=1, alpha=0.6)
    ax.text(90.5, ymax * 0.92, 'FT', color=C_MUTED, fontsize=7)

    ax.set_xlabel('Minute', color=C_MUTED, fontsize=9)
    ax.set_ylabel('Cumulative xG', color=C_MUTED, fontsize=9)
    ax.set_title('XG FLOW', color=C_MUTED, fontsize=9, pad=8,
                 fontweight='bold', loc='left', x=0.04)
    ax.tick_params(colors=C_MUTED, labelsize=8)
    ax.legend(facecolor=C_CARD, edgecolor=C_BORDER, labelcolor=C_TEXT, fontsize=8)
    ax.grid(axis='y', color=C_BORDER, alpha=0.4, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color(C_BORDER)
    ax.spines['left'].set_color(C_BORDER)


# ── GK spotlight ──────────────────────────────────────────────────────────────
def plot_gk(events, team1, team2, ax):
    gk_match = events[(events['type'] == 'Goal Keeper') & (events['period'] != 5)]

    saves_count = (gk_match[gk_match['goalkeeper_type'] == 'Shot Saved']
                   .groupby(['team', 'player'])['id'].count()
                   .reset_index().rename(columns={'id': 'saves'}))
    goals_conceded = (gk_match[gk_match['goalkeeper_type'] == 'Goal Conceded']
                      .groupby('team')['id'].count()
                      .reset_index().rename(columns={'id': 'goals_conceded'}))
    psxg = (events[(events['type'] == 'Shot') & (events['period'] != 5) &
                   (events['shot_outcome'].isin(['Goal', 'Saved']))]
            .groupby('team')['shot_statsbomb_xg'].sum().round(2)
            .reset_index().rename(columns={'shot_statsbomb_xg': 'psxg_faced'}))
    pen_saved = (events[(events['period'] == 5) &
                        (events['type'] == 'Goal Keeper') &
                        (events['goalkeeper_type'] == 'Shot Saved')]
                 .groupby('team')['id'].count()
                 .reset_index().rename(columns={'id': 'penalties_saved_against'}))

    gk = (saves_count
          .merge(goals_conceded, on='team', how='left')
          .merge(psxg, on='team', how='left')
          .merge(pen_saved, on='team', how='left')
          .fillna(0))
    gk['psxg_prevented'] = (gk['psxg_faced'] - gk['goals_conceded']).round(2)
    gk = gk[gk['team'].isin([team1, team2])].reset_index(drop=True)

    _panel(ax, 'GK SPOTLIGHT', accent=C_MUTED)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    team_colors = {team1: C_ARG, team2: C_FRA}
    slot_ys = [0.68, 0.22]

    for i, (_, row) in enumerate(gk.iterrows()):
        if i >= len(slot_ys):
            break
        y = slot_ys[i]
        c = team_colors.get(row['team'], C_TEXT)

        ax.plot([0.04, 0.065], [y + 0.15, y + 0.15],
                color=c, linewidth=3.5, solid_capstyle='round',
                transform=ax.transAxes)
        ax.text(0.09, y + 0.13, row['player'],
                color=C_TEXT, fontsize=9, fontweight='bold',
                transform=ax.transAxes)
        ax.text(0.09, y + 0.02,
                f"Saves {int(row['saves'])}   "
                f"Conceded {int(row['goals_conceded'])}   "
                f"PSxG prevented {row['psxg_prevented']:+.2f}",
                color=c, fontsize=8, transform=ax.transAxes)
        pen_txt = (f"Shootout saves: {int(row['penalties_saved_against'])}"
                   if row['penalties_saved_against'] > 0 else 'No shootout saves')
        ax.text(0.09, y - 0.10, pen_txt,
                color=C_MUTED, fontsize=8, transform=ax.transAxes)

        if i == 0:
            ax.axhline(y=0.44, xmin=0.04, xmax=0.96,
                       color=C_BORDER, linewidth=0.8, transform=ax.transAxes)


# ── Top 3 players ─────────────────────────────────────────────────────────────
def get_top3(events, team):
    shots = events[(events['type'] == 'Shot') & (events['period'] != 5)]
    player_xg = (shots.groupby(['team', 'player'])['shot_statsbomb_xg']
                 .sum().round(2).reset_index()
                 .rename(columns={'shot_statsbomb_xg': 'xg'}))
    goals = (events[events['shot_outcome'] == 'Goal']
             .groupby(['team', 'player'])['id'].count()
             .reset_index().rename(columns={'id': 'goals'}))
    assists = (events[events['pass_goal_assist'] == True]
               .groupby(['team', 'player'])['id'].count()
               .reset_index().rename(columns={'id': 'assists'}))
    top = (player_xg
           .merge(goals,   on=['team', 'player'], how='left')
           .merge(assists, on=['team', 'player'], how='left')
           .fillna(0))
    return (top[top['team'] == team]
            .sort_values(['goals', 'xg'], ascending=False)
            .head(3).reset_index(drop=True))


def plot_top_players(ax, top3, color, title):
    _panel(ax, title, accent=color)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    badge_colors = [C_GOLD, C_SILVER, C_BRONZE]
    slot_ys = [0.78, 0.50, 0.22]

    for i, (row, y) in enumerate(zip(top3.itertuples(), slot_ys)):
        badge = mpatches.FancyBboxPatch(
            (0.035, y + 0.015), 0.058, 0.060,
            boxstyle='round,pad=0.008',
            linewidth=0, facecolor=badge_colors[i],
            transform=ax.transAxes, clip_on=False, zorder=3)
        ax.add_patch(badge)
        ax.text(0.064, y + 0.045, str(i + 1),
                color='#0d1117', fontsize=8, fontweight='bold',
                ha='center', va='center', transform=ax.transAxes, zorder=4)
        ax.text(0.16, y + 0.065, row.player,
                color=C_TEXT, fontsize=9, fontweight='bold',
                transform=ax.transAxes)
        ax.text(0.16, y - 0.045,
                f"xG {row.xg:.2f}   G {int(row.goals)}   A {int(row.assists)}",
                color=color, fontsize=8.5, transform=ax.transAxes)
        if i < 2:
            ax.axhline(y=y - 0.105, xmin=0.04, xmax=0.96,
                       color=C_BORDER, linewidth=0.5, transform=ax.transAxes)


# ── Shot map ──────────────────────────────────────────────────────────────────
def plot_shotmap(events, team1, team2, ax):
    df = (events[events['type'] == 'Shot']
          [['team', 'shot_outcome', 'shot_statsbomb_xg', 'location', 'shot_type']]
          .copy())
    df = df[df['shot_type'] != 'Penalty'].copy()
    df['x']  = df['location'].apply(lambda l: l[0])
    df['y']  = df['location'].apply(lambda l: l[1])
    df['xg'] = df['shot_statsbomb_xg'].fillna(0).clip(lower=0)
    df['sz'] = (df['xg'] * 500).clip(lower=50)

    pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#30363d')
    pitch.draw(ax=ax)

    groups = {
        (team1, True):  (df[(df['team'] == team1) & (df['shot_outcome'] == 'Goal')],  C_ARG, False),
        (team1, False): (df[(df['team'] == team1) & (df['shot_outcome'] != 'Goal')],  C_ARG, False),
        (team2, True):  (df[(df['team'] == team2) & (df['shot_outcome'] == 'Goal')],  C_FRA, True),
        (team2, False): (df[(df['team'] == team2) & (df['shot_outcome'] != 'Goal')],  C_FRA, True),
    }
    for (_, is_goal), (sub, c, flip) in groups.items():
        if sub.empty:
            continue
        xs = (120 - sub['x']) if flip else sub['x']
        ys = (80  - sub['y']) if flip else sub['y']
        pitch.scatter(x=xs, y=ys, ax=ax, s=sub['sz'], zorder=3,
                      ec=c, c=c if is_goal else 'none',
                      linewidths=1.5, alpha=0.85 if is_goal else 0.50)

    t1a, t2a = team1[:3].upper(), team2[:3].upper()
    ax.scatter([], [], s=70, ec=C_ARG, c=C_ARG,  linewidths=1.5, label=f'{t1a} goal')
    ax.scatter([], [], s=70, ec=C_ARG, c='none', linewidths=1.5, label=f'{t1a} shot')
    ax.scatter([], [], s=70, ec=C_FRA, c=C_FRA,  linewidths=1.5, label=f'{t2a} goal')
    ax.scatter([], [], s=70, ec=C_FRA, c='none', linewidths=1.5, label=f'{t2a} shot')
    ax.legend(facecolor=C_CARD, edgecolor=C_BORDER, labelcolor=C_TEXT,
              fontsize=7, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.01))


# ─────────────────────────────────────────────────────────────────────────────
#  FIGURE
# ─────────────────────────────────────────────────────────────────────────────
plt.style.use('dark_background')
fig = plt.figure(figsize=(22, 18), facecolor=C_BG)

gs = GridSpec(
    4, 3, figure=fig,
    height_ratios=[0.65, 0.32, 3.0, 2.8],
    hspace=0.38, wspace=0.26,
    top=0.94, bottom=0.04, left=0.05, right=0.97
)

# ── Header ────────────────────────────────────────────────────────────────────
ax_header = fig.add_subplot(gs[0, :])
ax_header.set_facecolor(C_BG)
ax_header.set_xticks([])
ax_header.set_yticks([])
for s in ax_header.spines.values():
    s.set_visible(False)

score_bg = FancyBboxPatch(
    (0.22, 0.04), 0.56, 0.92, boxstyle='round,pad=0.02',
    linewidth=1, edgecolor=C_BORDER, facecolor=C_CARD,
    transform=ax_header.transAxes, clip_on=False)
ax_header.add_patch(score_bg)

ax_header.text(0.29, 0.54, 'Argentina', ha='center', va='center',
               fontsize=15, color=C_ARG, fontweight='bold',
               transform=ax_header.transAxes)
ax_header.text(0.50, 0.54, '3  -  3', ha='center', va='center',
               fontsize=24, color=C_TEXT, fontweight='bold',
               transform=ax_header.transAxes)
ax_header.text(0.71, 0.54, 'France', ha='center', va='center',
               fontsize=15, color=C_FRA, fontweight='bold',
               transform=ax_header.transAxes)
ax_header.text(0.5, 0.13,
               'FIFA World Cup Final 2022  -  Lusail Stadium  -  18 Dec 2022',
               ha='center', va='center', fontsize=8.5, color=C_MUTED,
               transform=ax_header.transAxes)

# ── Stat boxes ────────────────────────────────────────────────────────────────
def _stat_box(ax, label, val_str):
    ax.set_facecolor(C_CARD)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    rect = mpatches.FancyBboxPatch(
        (0.01, 0.01), 0.98, 0.98, boxstyle='round,pad=0.01',
        linewidth=1, edgecolor=C_BORDER, facecolor='none',
        transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    ax.text(0.5, 0.67, label, ha='center', va='center',
            fontsize=8, color=C_MUTED, fontweight='bold',
            transform=ax.transAxes)
    ax.text(0.5, 0.28, val_str, ha='center', va='center',
            fontsize=12, color=C_TEXT, fontweight='bold',
            transform=ax.transAxes)

ax_stat1 = fig.add_subplot(gs[1, 0])
_stat_box(ax_stat1, 'EXPECTED GOALS (xG)',
          f"ARG  {xg.get('Argentina', 0):.2f}  -  {xg.get('France', 0):.2f}  FRA")

ax_stat2 = fig.add_subplot(gs[1, 1])
_stat_box(ax_stat2, 'POSSESSION',
          f"ARG  {possession_pct.get('Argentina', 0):.1f}%  -  {possession_pct.get('France', 0):.1f}%  FRA")

ax_stat3 = fig.add_subplot(gs[1, 2])
_stat_box(ax_stat3, 'SHOTS ON TARGET',
          f"ARG  {shots_on_target.get('Argentina', 0)}  -  {shots_on_target.get('France', 0)}  FRA")

# ── Row 2 — top players / shot map ───────────────────────────────────────────
ax_left  = fig.add_subplot(gs[2, 0])
top3_arg = get_top3(events, 'Argentina')
plot_top_players(ax_left, top3_arg, C_ARG, 'TOP PLAYERS - ARG')

ax_center = fig.add_subplot(gs[2, 1])
ax_center.set_facecolor(C_BG)
ax_center.set_title('SHOT MAP', color=C_MUTED, fontsize=9, pad=8,
                     fontweight='bold', loc='left', x=0.04)
plot_shotmap(events, team1='Argentina', team2='France', ax=ax_center)

ax_right = fig.add_subplot(gs[2, 2])
top3_fra = get_top3(events, 'France')
plot_top_players(ax_right, top3_fra, C_FRA, 'TOP PLAYERS - FRA')

# ── Row 3 — stats / xG flow / GK ─────────────────────────────────────────────
ax_bl = fig.add_subplot(gs[3, 0])
plot_match_stats(events, team1='Argentina', team2='France', ax=ax_bl)

ax_bc = fig.add_subplot(gs[3, 1])
plot_xgflow(events, team1='Argentina', team2='France', ax=ax_bc)

ax_br = fig.add_subplot(gs[3, 2])
plot_gk(events, team1='Argentina', team2='France', ax=ax_br)

fig.suptitle('MATCH DASHBOARD', fontsize=12, color=C_MUTED,
             fontweight='bold', y=0.995)

plt.savefig('argentina_france_dashboard.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()
print('Dashboard ready.')
